# Tosh, Qaychi, Qog'oz

**Tosh, Qaychi, Qog'oz** (Rock, Scissors, Paper) is a game deeply embedded in Mongolian culture —
played to resolve disputes, make decisions, and just have fun.
In this miniapp, your webcam becomes the game controller: hold up a hand gesture and the computer picks its move.

<a target="_blank" href="https://jupyter-k12.org?github=simonguest/codercub/blob/main/labs/07/notebooks/tosh_qaychi_qogoz.ipynb">
  <img src="https://img.shields.io/badge/Open_in-Jupyter_K--12-blue" alt="Open In Jupyter K-12"/>
</a>

## How It Works

Three gestures map to the three moves:

| Gesture | Mongolian | Meaning |
|---|---|---|
| Closed fist | **Tosh** | Rock |
| V-sign (Victory) | **Qaychi** | Scissors |
| Open palm | **Qog'oz** | Paper |

The rules are the same as always:
- **Tosh** crushes **Qaychi**
- **Qaychi** cuts **Qog'oz**
- **Qog'oz** covers **Tosh**

Hold a gesture steady for about **1 second** to register your play.
A charge bar shows your progress while you hold, and there's a short pause between rounds so you can see the result.

## Step 1: Practice Your Gestures

Run the cell below to see what the detector recognizes for each gesture.
Try all three moves and check the output to confirm they're being picked up correctly.

Click **Stop** (■) when you're ready to play.

In [ ]:
import cv
import graphics
import time

canvas = graphics.canvas()
camera = cv.start_camera(canvas)
detector = cv.start_gesture_detector(camera)

VALID = {'Closed_Fist', 'Victory', 'Open_Palm'}

frame = 0
try:
    while True:
        detections = detector.get_detections()
        canvas.draw_hands(detections)
        if frame % 30 == 0:
            for hand in detections:
                if hand['gesture'] in VALID:
                    print(f"  {hand['handedness']}: {hand['gesture']} ({hand['confidence']:.0%})")
        frame += 1
        time.sleep(0.033)
finally:
    detector.stop()
    camera.stop()
    print("Camera stopped.")

## Step 2: Play!

Run the cell below and hold any of the three gestures steady for about 1 second.
The charge bar fills as you hold — when it's full, the computer makes its move and the result appears.
There's a short pause between rounds so you can read the result before the next round starts.

Your score carries across rounds until you click **Stop** (■).

In [ ]:
import cv
import graphics
import time
import random

VALID        = {'Closed_Fist': 'Tosh', 'Victory': 'Qaychi', 'Open_Palm': "Qog'oz"}
BEATS        = {'Closed_Fist': 'Victory', 'Victory': 'Open_Palm', 'Open_Palm': 'Closed_Fist'}
HOLD         = 40   # frames to hold (~1.3 sec at 30fps)
PAUSE_FRAMES = 60   # pause between rounds (~2 sec)

canvas = graphics.canvas()
camera = cv.start_camera(canvas)
detector = cv.start_gesture_detector(camera)

held, count = None, 0
result_line = "Hold a gesture steady to play!"
score = [0, 0]
pause = 0

try:
    while True:
        detections = detector.get_detections()
        canvas.draw_hands(detections)

        if pause > 0:
            pause -= 1
        else:
            gesture = detections[0]['gesture'] if detections else None
            if gesture in VALID:
                count = count + 1 if gesture == held else 1
                held  = gesture
                if count >= HOLD:
                    comp = random.choice(list(VALID))
                    if gesture == comp:
                        msg = "Tie!"
                    elif BEATS[gesture] == comp:
                        msg = "You win!"
                        score[0] += 1
                    else:
                        msg = "Computer wins!"
                        score[1] += 1
                    result_line = f"{VALID[gesture]}  vs  {VALID[comp]}  —  {msg}"
                    held, count = None, 0
                    pause = PAUSE_FRAMES
            else:
                held, count = None, 0

        ctx = canvas.get_context('2d')
        ctx.fill_style = 'rgba(0,0,0,0.7)'
        ctx.fill_rect(0, 0, 800, 84)
        ctx.fill_style = '#ffffff'
        ctx.font = 'bold 22px sans-serif'
        ctx.fill_text(result_line, 12, 28)
        ctx.font = '18px sans-serif'
        ctx.fill_text(f"You: {score[0]}   Computer: {score[1]}", 12, 54)
        if pause > 0:
            ctx.fill_text("Get ready...", 12, 78)
        elif held:
            bars = int(count / HOLD * 10)
            ctx.fill_text(f"Holding {VALID[held]}... {'#' * bars}{'.' * (10 - bars)}", 12, 78)

        time.sleep(0.033)
finally:
    detector.stop()
    camera.stop()
    print("Camera stopped.")

What would you add or change to make this game more fun? (Think about sound effects, a countdown, a best-of-3 mode, or showing the computer's move as an image.)

## Ideas to Extend This App

- **Sound effects** — play a chime on a win using `import audio; audio.play('/sample_files/chime.wav')`
- **Countdown** — show "3... 2... 1... Show!" on the canvas before reading the gesture
- **Best of 3** — track rounds and announce an overall winner after 3 games
- **Two-player mode** — use both hands (`num_hands=2`) and compare them directly
- **Win streak** — track and display the longest winning streak